# Hybrid BM25 + Dense Retrieval RAG with ChromaDB

This notebook demonstrates a **hybrid retrieval** Retrieval-Augmented Generation (RAG) pipeline combining:
- **BM25** (sparse, keyword-based retrieval) via `rank_bm25`
- **Dense vector retrieval** via ChromaDB with sentence-transformers embeddings
- **Reciprocal Rank Fusion (RRF)** to merge results from both retrievers

Hybrid retrieval consistently outperforms either method alone, especially on multi-hop and factoid QA tasks. This pattern is used in production RAG systems to balance lexical precision with semantic recall.

**Requirements:**
```
pip install chromadb rank_bm25 sentence-transformers
```

In [ ]:
# Install dependencies
# !pip install chromadb rank_bm25 sentence-transformers

In [ ]:
import chromadb
from chromadb.utils import embedding_functions
from rank_bm25 import BM25Okapi
import re
from typing import List, Dict, Tuple

## 1. Sample Corpus

We use a small corpus of factual passages for demonstration. In production, replace this with your document chunks.

In [ ]:
corpus = [
    "ChromaDB is an open-source vector database designed for AI applications and LLM-powered systems.",
    "Retrieval-Augmented Generation (RAG) combines a retriever with a language model to answer questions grounded in a knowledge base.",
    "BM25 is a probabilistic ranking function used in information retrieval, effective for keyword-heavy queries.",
    "Dense retrieval uses neural embeddings to find semantically similar documents even when keywords don't overlap.",
    "Reciprocal Rank Fusion (RRF) merges ranked lists from multiple retrievers by summing reciprocal ranks.",
    "Sentence transformers produce fixed-size embeddings from variable-length text using transformer encoder models.",
    "Hybrid search combines sparse and dense retrieval to improve recall and precision over either method alone.",
    "HotPotQA is a multi-hop question answering dataset requiring reasoning over multiple documents.",
    "Cross-encoder reranking re-scores retrieved candidates using a model that jointly encodes the query and document.",
    "Vector databases store and index high-dimensional embeddings, enabling fast approximate nearest-neighbor search.",
]

doc_ids = [f"doc_{i}" for i in range(len(corpus))]

## 2. Set Up ChromaDB Collection (Dense Retriever)

In [ ]:
# Use a local sentence-transformer model — no API key needed
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

client = chromadb.Client()
collection = client.get_or_create_collection(
    name="hybrid_rag_demo",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)

# Index documents
collection.add(
    ids=doc_ids,
    documents=corpus,
)

print(f"Indexed {collection.count()} documents into ChromaDB.")

## 3. Set Up BM25 (Sparse Retriever)

In [ ]:
def tokenize(text: str) -> List[str]:
    """Simple whitespace + lowercase tokenizer for BM25."""
    return re.sub(r"[^a-z0-9\s]", "", text.lower()).split()

tokenized_corpus = [tokenize(doc) for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index built.")

## 4. Hybrid Retrieval with Reciprocal Rank Fusion (RRF)

In [ ]:
def dense_retrieve(query: str, top_k: int = 5) -> List[Tuple[str, float]]:
    """Retrieve top-k documents using ChromaDB dense search."""
    results = collection.query(query_texts=[query], n_results=top_k)
    ids = results["ids"][0]
    distances = results["distances"][0]
    # Convert cosine distance to similarity score
    return [(doc_id, 1 - dist) for doc_id, dist in zip(ids, distances)]


def bm25_retrieve(query: str, top_k: int = 5) -> List[Tuple[str, float]]:
    """Retrieve top-k documents using BM25 sparse search."""
    tokenized_query = tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]
    return [(doc_ids[idx], score) for idx, score in ranked]


def reciprocal_rank_fusion(
    ranked_lists: List[List[Tuple[str, float]]],
    k: int = 60
) -> List[Tuple[str, float]]:
    """
    Merge multiple ranked lists using Reciprocal Rank Fusion.
    RRF score = sum(1 / (k + rank)) across all lists.
    k=60 is the standard default from the original RRF paper.
    """
    rrf_scores: Dict[str, float] = {}
    for ranked in ranked_lists:
        for rank, (doc_id, _) in enumerate(ranked, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)


def hybrid_retrieve(query: str, top_k: int = 5) -> List[Dict]:
    """Full hybrid retrieval: BM25 + dense via ChromaDB, fused with RRF."""
    dense_results = dense_retrieve(query, top_k=top_k)
    bm25_results = bm25_retrieve(query, top_k=top_k)

    fused = reciprocal_rank_fusion([dense_results, bm25_results])

    # Map back to documents
    id_to_doc = dict(zip(doc_ids, corpus))
    return [
        {"id": doc_id, "score": score, "document": id_to_doc[doc_id]}
        for doc_id, score in fused[:top_k]
    ]

## 5. Run Example Queries

In [ ]:
queries = [
    "How does hybrid search improve retrieval?",
    "What is BM25 and when should I use it?",
    "How do vector databases work with embeddings?",
]

for query in queries:
    print(f"\nQuery: {query}")
    print("-" * 60)
    results = hybrid_retrieve(query, top_k=3)
    for i, res in enumerate(results, 1):
        print(f"  {i}. [score={res['score']:.4f}] {res['document']}")

## 6. Compare: Dense-Only vs BM25-Only vs Hybrid

The table below illustrates how each retriever ranks results for a keyword-heavy query.

In [ ]:
query = "BM25 keyword ranking function"
print(f"Query: '{query}'\n")

print("Dense-only (ChromaDB):")
for rank, (doc_id, score) in enumerate(dense_retrieve(query, top_k=3), 1):
    idx = doc_ids.index(doc_id)
    print(f"  {rank}. {corpus[idx][:80]}...")

print("\nBM25-only (sparse):")
for rank, (doc_id, score) in enumerate(bm25_retrieve(query, top_k=3), 1):
    idx = doc_ids.index(doc_id)
    print(f"  {rank}. {corpus[idx][:80]}...")

print("\nHybrid (RRF fusion):")
for rank, res in enumerate(hybrid_retrieve(query, top_k=3), 1):
    print(f"  {rank}. {res['document'][:80]}...")

## 7. Key Takeaways

| Method | Strength | Weakness |
|---|---|---|
| BM25 (sparse) | Exact keyword matches, fast | Misses semantic similarity |
| Dense (ChromaDB) | Semantic understanding | Can miss exact terms |
| Hybrid (RRF) | Best of both worlds | Slightly more compute |

**When to use hybrid retrieval:**
- Multi-hop QA where queries use specific entity names (BM25 helps)
- Domain-specific corpora with technical terminology
- Any production RAG system where recall matters

**Further improvements:**
- Add a cross-encoder reranker on the fused top-k results
- Tune the RRF `k` parameter for your corpus
- Use ChromaDB's metadata filtering to pre-filter before retrieval